# Семинар 2. Гомография - nbjw
Подготовил: Дурыгин Олег

Данные: отрезок 30 секунд из матча из датасета Soccernet GSR

## 1. Installation and download models

In [1]:
# Install dependencies (only in Colab; no repo clone)
! wget https://github.com/mguti97/No-Bells-Just-Whistles/releases/download/v1.0.0/SV_kp
! wget https://github.com/mguti97/No-Bells-Just-Whistles/releases/download/v1.0.0/SV_lines
! git clone https://github.com/mguti97/No-Bells-Just-Whistles.git
! cd No-Bells-Just-Whistles && pip install -r requirements.txt
print("Installation done.")

--2026-02-27 15:36:51--  https://github.com/mguti97/No-Bells-Just-Whistles/releases/download/v1.0.0/SV_kp
Resolving github.com (github.com)... 140.82.114.4
Connecting to github.com (github.com)|140.82.114.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/784307854/e6f5992d-750d-4e19-9ba9-1b00818fdec5?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-02-27T16%3A30%3A04Z&rscd=attachment%3B+filename%3DSV_kp&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-02-27T15%3A29%3A58Z&ske=2026-02-27T16%3A30%3A04Z&sks=b&skv=2018-11-09&sig=XmU9%2BuY0dBWTTHeNq62wTGq%2BEjbpqV9qUOza1G7GQ3c%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc3MjIxMDIxMSwibmJmIjoxNzcyMjA2NjExLCJwYXRoIjoicmVsZWFzZWFzc2V0cHJvZHVjdGlvbi5ibG9iLmN

## 2. Load frames from a zip (e.g. on Google Drive)

In Colab: mount Google Drive and set `ZIP_PATH` to your zip of frames (e.g. `"/content/drive/MyDrive/frames.zip"`). Frames are extracted and loaded in **logical order** (natural sort by filename: frame_001.jpg, frame_002.jpg, …).

In [2]:
!gdown 1R3lTyOlCYoQb5EOcrMIQ4yJn0_z1lYhS

Downloading...
From (original): https://drive.google.com/uc?id=1R3lTyOlCYoQb5EOcrMIQ4yJn0_z1lYhS
From (redirected): https://drive.google.com/uc?id=1R3lTyOlCYoQb5EOcrMIQ4yJn0_z1lYhS&confirm=t&uuid=f2f4b1f2-854e-493b-b019-2ed3c11698df
To: /content/SNGS-141.zip
100% 171M/171M [00:02<00:00, 62.9MB/s]


In [3]:
import re
import zipfile
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

# Natural sort
def _natural_sort_key(s: str):
    return [int(x) if x.isdigit() else x.lower() for x in re.split(r"(\d+)", s)]

IMAGE_EXTENSIONS = [".jpg", ".png"]
ZIP_PATH = "SNGS-141.zip"
EXTRACT_DIR = Path("/content/frames")
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(EXTRACT_DIR)

# List images and sort logically
image_paths = []
for p in EXTRACT_DIR.rglob("*"):
    if p.suffix.lower() in IMAGE_EXTENSIONS:
        image_paths.append(p)
image_paths = sorted(image_paths, key=lambda p: _natural_sort_key(p.name))

# Load as RGB in order
frames = []
for p in image_paths:
    img = cv2.imread(str(p))
    if img is not None:
        frames.append(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

print(f"Loaded {len(frames)} frames from {ZIP_PATH} (order: {[p.name for p in image_paths[:3]]} ...)")
if not frames:
    raise FileNotFoundError(f"No images found in {ZIP_PATH}. Use a zip of .jpg/.png frames.")

Loaded 750 frames from SNGS-141.zip (order: ['000001.jpg', '000002.jpg', '000003.jpg'] ...)


## 3. Run detection on a single frame

Read one frame and run the detector. Results are Ultralytics `Results` (use `result.boxes`: xyxy, conf, cls).

In [8]:
! cd /content/No-Bells-Just-Whistles/ && python inference.py \
  --weights_kp /content/SV_kp \
  --weights_line /content/SV_lines \
  --input_path /content/frames/SNGS-141/img1/000200.jpg \
  --input_type "image" \
  --save_path /content/test_result.png

## 4. Visualize field detection

In [9]:
from IPython.display import Image

Image("/content/test_result.png")

Output hidden; open in https://colab.research.google.com to view.